# 5.2.3 · Verificação da qualidade dos registros
## Total e percentual de polígonos inválidos por ano (2021–2025)

Este notebook audita as geometrias **como foram fornecidas**, sem corrigir, reprojetar, recortar ou eliminar registros repetidos. Cada linha com `Polygon` ou `MultiPolygon` conta como **um registro poligonal**, independentemente do número de partes. Não há deduplicação por `REF_BACEN`, `ID_GLEBA` ou `id_gleba2`.

**Use os arquivos anteriores à correção das geometrias** para responder quantos casos eram originalmente inválidos. Uma Base02/Base04 já corrigida só permite avaliar sua condição atual. O resultado não comprova, por si só, quantos casos foram corrigidos ou recuperados.

### O que será gerado
- Tabela por ano e total geral: registros, polígonos avaliados, polígonos inválidos e percentuais.
- Contagens separadas de geometrias ausentes, vazias e de tipos não poligonais.
- Relação dos registros problemáticos e motivos de invalidade.
- CSVs compatíveis com Excel, tabela HTML para copiar ao Word e texto de apoio com os resultados.
- ZIP com todos os resultados.

### Denominadores explícitos
**Percentual principal = 100 × polígonos inválidos / polígonos não vazios avaliados no ano.**

Também é informado o percentual de polígonos inválidos em relação a **todas as linhas** do ano. Geometrias nulas, vazias e pontos/linhas não são classificadas como polígonos inválidos. Anos sem registros têm percentual indefinido, exibido como “—”, e não 0%.

### Recorte e ano
Utilize o mesmo universo da seção 5.2.3. Se carregar a base nacional, os resultados serão nacionais. Este notebook **não executa recorte do Cerrado**, porque isso poderia alterar a geometria antes da auditoria. Para auditar o recorte da pesquisa, use as geometrias originais dos registros selecionados para esse recorte.

Informe se o ano é o ano do arquivo, da contratação ou outra referência. Em arquivos anuais, o ano é atribuído explicitamente na configuração. Em uma base consolidada, informe a coluna de ano ou de data.


## 1. Instalar as bibliotecas

In [ ]:
%pip -q install "geopandas>=1.0,<2" "shapely>=2.0,<3" "pandas>=2.2,<3" "pyarrow>=14"

## 2. Acessar o Google Drive (opcional)
Se seus arquivos estiverem no Drive, execute a célula e autorize o acesso. Se fizer upload pelo painel Arquivos do Colab, defina `USAR_DRIVE = False`.

In [ ]:
USAR_DRIVE = True
if USAR_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        print("Execução fora do Colab: use caminhos locais na configuração.")
    else:
        drive.mount("/content/drive")


## 3. Configurar os arquivos
**Edite somente esta célula.** Aceita `.gpkg`, `.shp`, ZIP contendo Shapefile, `.geojson`, `.json` e GeoParquet (`.parquet`). Para SHP, mantenha juntos os arquivos auxiliares. Para ZIP com vários Shapefiles ou GPKG com várias camadas, preencha `camada`.

**Opção A — um arquivo por ano:** preencha os cinco caminhos abaixo e mantenha os anos correspondentes. Os caminhos são dos arquivos existentes, não de pastas.

**Opção B — base consolidada:** substitua a lista FONTES por uma única entrada como:
```python
FONTES = [{"caminho": "/content/drive/MyDrive/pesquisa/base_original.gpkg", "ano": None, "camada": None}]
COLUNA_ANO = "ANO"  # informe o nome exato; pode ser uma coluna numérica ou textual
```
Se houver apenas uma coluna de data, use `COLUNA_ANO = None`, `COLUNA_DATA = "DT_EMISSAO"` e o formato correto, por exemplo `%d/%m/%Y` ou `%Y-%m-%d`. O ano **não é inferido automaticamente pelo nome do arquivo**.

Para ver os arquivos de uma pasta, use opcionalmente: `print(*Path("/content/drive/MyDrive/sua_pasta").glob("*"), sep="\n")`.


In [ ]:
from pathlib import Path
FONTES = [
    {"caminho": "", "ano": 2021, "camada": None},
    {"caminho": "", "ano": 2022, "camada": None},
    {"caminho": "", "ano": 2023, "camada": None},
    {"caminho": "", "ano": 2024, "camada": None},
    {"caminho": "", "ano": 2025, "camada": None},
]
COLUNA_ANO = None
COLUNA_DATA = None
FORMATO_DATA = "%d/%m/%Y"
COLUNAS_IDENTIFICADORAS = ["ID_GLEBA", "id_gleba2", "REF_BACEN", "NU_ORDEM"]
DESCRICAO_UNIVERSO = "PREENCHER: informe se são registros do Cerrado ou de abrangência nacional"
REFERENCIA_ANO = "Ano do arquivo de origem"  # ajuste se usar ano de emissão/contratação
ESTAGIO_BASE = "não informado"  # "antes das correções", "após tratamento" ou "não informado"
SAIDA = Path("/content/auditoria_geometrias") if Path("/content").exists() else Path.cwd()/"auditoria_geometrias"
BAIXAR_ZIP = True
ANOS = list(range(2021, 2026))


## 4. Funções de leitura e auditoria
A validade topológica é verificada por `shapely.is_valid`, e o motivo por `shapely.is_valid_reason`. A verificação considera a geometria em 2D, não a precisão posicional, sobreposições entre registros ou a adequação jurídica dos limites.

Documentação: [is_valid](https://shapely.readthedocs.io/en/stable/reference/shapely.is_valid.html) e [is_valid_reason](https://shapely.readthedocs.io/en/stable/reference/shapely.is_valid_reason.html).

Erros de leitura/decodificação interrompem a execução, em vez de gerar uma estatística parcial. Nenhuma opção de reparação automática é ativada. Os IDs são preservados no relatório como texto, mas zeros já perdidos no arquivo de origem não podem ser recuperados.


In [ ]:
import json, zipfile, hashlib, shutil, platform, warnings, html
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from IPython.display import display, HTML

CATEGORIAS = ["poligono_valido", "poligono_invalido", "geometria_ausente", "geometria_vazia", "tipo_nao_poligonal"]

def ler(fonte):
    p = Path(fonte["caminho"])
    camada = fonte.get("camada")
    if not p.is_file(): raise FileNotFoundError(f"Arquivo não encontrado: {p}")
    uri = str(p)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as z:
            membros = [m for m in z.namelist() if m.lower().endswith(".shp") and not m.startswith("__MACOSX")]
        if camada:
            membros = [m for m in membros if m == camada or Path(m).stem == camada]
        if len(membros) != 1:
            raise ValueError(f"ZIP {p.name}: informe camada para selecionar um Shapefile. Opções: {membros}")
        uri = "zip://" + str(p.resolve()) + "!" + membros[0]
    elif p.suffix.lower() == ".gpkg":
        nomes = gpd.list_layers(p)["name"].tolist()
        if camada is None and len(nomes) != 1:
            raise ValueError(f"GPKG {p.name}: informe a camada. Opções: {nomes}")
    with warnings.catch_warnings(record=True) as capturadas:
        warnings.simplefilter("always")
        if p.suffix.lower() in (".parquet", ".geoparquet"):
            gdf = gpd.read_parquet(p)
        else:
            gdf = gpd.read_file(uri, **({"layer":camada} if camada and p.suffix.lower() != ".zip" else {}))
    avisos = [str(w.message) for w in capturadas]
    for aviso in avisos: print("Aviso de leitura:", aviso)
    if gdf.empty: print("Atenção: arquivo/camada sem registros:", p.name)
    return gdf.reset_index(drop=True), avisos

def anos_registros(gdf, fonte):
    ano = fonte.get("ano")
    if ano is not None:
        if ano not in ANOS: raise ValueError(f"Ano fixo fora do período: {ano}")
        return pd.Series(ano,index=gdf.index,dtype="Int64")
    if COLUNA_ANO:
        if COLUNA_ANO not in gdf: raise ValueError(f"Coluna {COLUNA_ANO!r} inexistente. Disponíveis: {list(gdf.columns)}")
        x = pd.to_numeric(gdf[COLUNA_ANO],errors="coerce")
        return x.where(x.notna() & np.isfinite(x) & x.mod(1).eq(0) & x.between(1,9999)).astype("Int64")
    if COLUNA_DATA:
        if COLUNA_DATA not in gdf: raise ValueError(f"Coluna {COLUNA_DATA!r} inexistente. Disponíveis: {list(gdf.columns)}")
        x = gdf[COLUNA_DATA]
        if pd.api.types.is_datetime64_any_dtype(x): datas=x
        else: datas=pd.to_datetime(x.astype("string"),format=FORMATO_DATA,errors="coerce")
        return datas.dt.year.astype("Int64")
    raise ValueError("Na base consolidada, informe COLUNA_ANO ou COLUNA_DATA, ou atribua um ano fixo ao arquivo.")

def auditar(gdf, fonte, numero_fonte):
    geom = gdf.geometry.array
    ausente = np.asarray(shapely.is_missing(geom))
    vazia = ~ausente & np.asarray(shapely.is_empty(geom))
    poligonal = gdf.geometry.geom_type.isin(["Polygon","MultiPolygon"]).to_numpy() & ~ausente & ~vazia
    valida = np.asarray(shapely.is_valid(geom))
    invalida = poligonal & ~valida
    status = np.full(len(gdf),"tipo_nao_poligonal",dtype=object)
    status[ausente] = "geometria_ausente"
    status[vazia] = "geometria_vazia"
    status[poligonal & valida] = "poligono_valido"
    status[invalida] = "poligono_invalido"
    motivo = np.full(len(gdf),"",dtype=object)
    motivo[invalida] = shapely.is_valid_reason(geom[invalida])
    motivo[ausente] = "Geometria ausente (NULL)"
    motivo[vazia] = "Geometria vazia (EMPTY)"
    outro = status == "tipo_nao_poligonal"
    motivo[outro] = "Tipo não poligonal: " + gdf.geometry.geom_type.fillna("desconhecido").to_numpy()[outro]
    df = pd.DataFrame({
        "fonte_numero":numero_fonte,"arquivo":str(Path(fonte["caminho"]).resolve()),
        "camada":fonte.get("camada") or "", "linha_leitura":np.arange(1,len(gdf)+1),
        "ano":anos_registros(gdf,fonte),"tipo_geometria":gdf.geometry.geom_type,
        "status":status,"motivo":motivo,
    })
    for c in COLUNAS_IDENTIFICADORAS:
        if c in gdf: df["id_"+c]=gdf[c].astype("string")
    return df

def resumir(auditoria):
    a=auditoria.loc[auditoria.ano.isin(ANOS)].copy()
    tab=pd.crosstab(a.ano,a.status).reindex(index=ANOS,columns=CATEGORIAS,fill_value=0)
    tab.index.name="ano"
    tab=tab.reset_index()
    total={"ano":"TOTAL",**{c:int(tab[c].sum()) for c in CATEGORIAS}}
    tab=pd.concat([tab,pd.DataFrame([total])],ignore_index=True)
    tab["registros_total"]=tab[CATEGORIAS].sum(axis=1)
    tab["poligonos_avaliados"]=tab.poligono_valido+tab.poligono_invalido
    tab["percentual_invalidos_sobre_poligonos"]=100*tab.poligono_invalido/tab.poligonos_avaliados.replace(0,np.nan)
    tab["percentual_invalidos_sobre_registros"]=100*tab.poligono_invalido/tab.registros_total.replace(0,np.nan)
    return tab[["ano","registros_total","poligonos_avaliados","poligono_valido","poligono_invalido",
                "percentual_invalidos_sobre_poligonos","percentual_invalidos_sobre_registros",
                "geometria_ausente","geometria_vazia","tipo_nao_poligonal"]]

def gravar_csv(df,nome):
    df.to_csv(SAIDA/nome,index=False,sep=";",decimal=",",encoding="utf-8-sig",float_format="%.6f")

def hash_arquivo(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()


## 5. Executar a auditoria
Cada registro permanece na contagem. Arquivos repetidos na configuração são rejeitados para evitar leitura duplicada acidental. Anos fora de 2021–2025 ficam em relatório separado. Registros com ano ausente ou ilegível impedem a tabela anual até que a configuração seja corrigida.

In [ ]:
if not FONTES or any(not str(f.get("caminho","")).strip() for f in FONTES):
    raise ValueError("Preencha os caminhos na célula 3. Para base consolidada, substitua as cinco entradas por uma única entrada.")
chaves=[(str(Path(f["caminho"]).resolve()),f.get("camada")) for f in FONTES]
if len(set(chaves))!=len(chaves):
    raise ValueError("A mesma fonte/camada foi indicada mais de uma vez. Carregue-a uma única vez.")
SAIDA.mkdir(parents=True,exist_ok=True)
# Remove apenas resultados anteriores deste notebook, evitando confundi-los com uma execução interrompida.
for nome in ["resumo_anual.csv","tabela_para_word.html","texto_resultados.txt","motivos_invalidade_por_ano.csv",
             "registros_problematicos.csv","registros_fora_periodo.csv","registros_sem_ano.csv","metadados.json"]:
    (SAIDA/nome).unlink(missing_ok=True)
partes=[]; entradas=[]
for i,fonte in enumerate(FONTES,1):
    print(f"Lendo {i}/{len(FONTES)}: {Path(fonte['caminho']).name}")
    gdf,avisos=ler(fonte)
    partes.append(auditar(gdf,fonte,i))
    entradas.append({**fonte,"registros_lidos":len(gdf),"crs":str(gdf.crs),"avisos_leitura":avisos,
                     "sha256_arquivo":hash_arquivo(fonte["caminho"])})
    del gdf
    print("Registros auditados:",len(partes[-1]))
auditoria=pd.concat(partes,ignore_index=True)
sem_ano=auditoria.loc[auditoria.ano.isna()].copy()
if len(sem_ano):
    gravar_csv(sem_ano,"registros_sem_ano.csv")
    raise ValueError(f"{len(sem_ano)} registros sem ano interpretável. Consulte registros_sem_ano.csv e corrija a coluna/formato antes de gerar os resultados anuais.")
fora=auditoria.loc[~auditoria.ano.isin(ANOS)].copy()
gravar_csv(fora,"registros_fora_periodo.csv")
base_periodo=auditoria.loc[auditoria.ano.isin(ANOS)].copy()
if base_periodo.empty: raise ValueError("Nenhum registro encontrado entre 2021 e 2025.")
resumo=resumir(base_periodo)
problemas=base_periodo.loc[base_periodo.status.ne("poligono_valido")].copy()
gravar_csv(problemas,"registros_problematicos.csv")
invalidos=base_periodo.loc[base_periodo.status.eq("poligono_invalido")].copy()
invalidos["classe_motivo"]=invalidos.motivo.str.replace(r"\[.*$","",regex=True).str.strip()
motivos=invalidos.groupby(["ano","classe_motivo"],dropna=False).size().reset_index(name="quantidade")
gravar_csv(motivos,"motivos_invalidade_por_ano.csv")
gravar_csv(resumo,"resumo_anual.csv")
assert int(resumo.iloc[-1].registros_total)==len(base_periodo)
assert int(resumo.iloc[-1].poligono_invalido)==len(invalidos)
print(f"Concluído: {len(base_periodo):,} registros no período; {len(fora):,} fora do período.")
if ESTAGIO_BASE != "antes das correções":
    print("Atenção: os resultados descrevem o estado dos arquivos enviados; não foram identificados como anteriores às correções.")
if any(e["avisos_leitura"] for e in entradas):
    print("Há avisos de leitura nos metadados: confira-os antes de utilizar os resultados.")


## 6. Tabela para responder ao orientador
O total percentual é calculado a partir das somas, e não pela média dos percentuais anuais. “—” significa denominador zero. A tabela inclui o denominador utilizado e os problemas de existência/tipo em colunas próprias.

In [ ]:
nomes={"ano":"Ano","registros_total":"Total de registros","poligonos_avaliados":"Polígonos avaliados",
"poligono_valido":"Polígonos válidos","poligono_invalido":"Polígonos inválidos",
"percentual_invalidos_sobre_poligonos":"Inválidos / polígonos (%)",
"percentual_invalidos_sobre_registros":"Inválidos / registros (%)",
"geometria_ausente":"Ausentes","geometria_vazia":"Vazias","tipo_nao_poligonal":"Não poligonais"}
def numero(x): return f"{int(x):,}".replace(",",".")
def percentual(x): return "—" if pd.isna(x) else f"{x:.2f}".replace(".",",")
tabela=resumo.copy()
for c in tabela.columns:
    if c.startswith("percentual_"): tabela[c]=tabela[c].map(percentual)
    elif c!="ano": tabela[c]=tabela[c].map(numero)
tabela=tabela.rename(columns=nomes)
nota=("Nota: cada registro Polygon ou MultiPolygon não vazio constitui uma unidade de avaliação. "
      "Percentual principal = polígonos inválidos / polígonos avaliados × 100. "
      "Ausentes, vazias e tipos não poligonais foram contabilizados separadamente. "
      "Não houve deduplicação. — indica ausência de denominador.")
html_tabela=tabela.to_html(index=False,border=0,escape=True)
pagina=("<!doctype html><html lang='pt-BR'><meta charset='utf-8'><title>Qualidade das geometrias</title>"
        "<style>body{font:14px Arial;margin:32px;color:#24382f}table{border-collapse:collapse;width:100%}"
        "th{background:#e6edde}th,td{padding:9px;border-bottom:1px solid #cad3c5;text-align:right}"
        "tr:last-child{font-weight:bold}p{line-height:1.5}</style>"
        "<h2>Qualidade das geometrias por ano — 2021–2025</h2>"
        f"<p>Universo: {html.escape(DESCRICAO_UNIVERSO)}<br>Referência temporal: {html.escape(REFERENCIA_ANO)}"
        f"<br>Estágio declarado: {html.escape(ESTAGIO_BASE)}</p>"+html_tabela+f"<p>{nota}</p></html>")
(SAIDA/"tabela_para_word.html").write_text(pagina,encoding="utf-8")
display(HTML(html_tabela))
print(nota)
print("Motivos de invalidade:")
display(motivos)


## 7. Texto de apoio e download
O texto descreve somente o diagnóstico observado; não afirma que todos os casos foram corrigidos. Confira o universo e a referência do ano antes de incorporar o texto ao subcapítulo.

In [ ]:
linhas=["Diagnóstico da qualidade geométrica dos registros fornecidos", "",
        f"Universo: {DESCRICAO_UNIVERSO}.",f"Referência temporal: {REFERENCIA_ANO}.",
        f"Estágio declarado da base: {ESTAGIO_BASE}.", ""]
for _,r in resumo.iterrows():
    rotulo="No conjunto de 2021 a 2025" if r.ano=="TOTAL" else f"Em {r.ano}"
    if r.poligonos_avaliados:
        linhas.append(f"{rotulo}, foram avaliados {numero(r.poligonos_avaliados)} registros poligonais não vazios, dos quais {numero(r.poligono_invalido)} eram inválidos ({percentual(r.percentual_invalidos_sobre_poligonos)}%).")
    else:
        linhas.append(f"{rotulo}, não havia registros poligonais não vazios para avaliação; o percentual não foi calculado.")
t=resumo.iloc[-1]
linhas.extend(["",f"Separadamente, foram contabilizadas {numero(t.geometria_ausente)} geometrias ausentes, {numero(t.geometria_vazia)} geometrias vazias e {numero(t.tipo_nao_poligonal)} registros de tipos não poligonais no período.",
              "A unidade de contagem foi o registro territorial, sem deduplicação por operação ou geometria. O percentual de invalidade foi calculado sobre os registros poligonais não vazios avaliados em cada ano.",
              "A auditoria descreve o estado dos arquivos fornecidos. Não mede precisão posicional nem comprova reparação das geometrias."])
if len(fora): linhas.append(f"Foram encontrados {numero(len(fora))} registros fora de 2021–2025, listados separadamente e não incluídos nos indicadores.")
texto="\n".join(linhas)
(SAIDA/"texto_resultados.txt").write_text(texto,encoding="utf-8")
print(texto)
meta={"data_utc":datetime.now(timezone.utc).isoformat(),"universo":DESCRICAO_UNIVERSO,
      "referencia_ano":REFERENCIA_ANO,"estagio_base_declarado":ESTAGIO_BASE,
      "fontes":entradas,"coluna_ano":COLUNA_ANO,"coluna_data":COLUNA_DATA,"formato_data":FORMATO_DATA,
      "registros_no_periodo":len(base_periodo),"registros_fora_periodo":len(fora),
      "denominador_principal":"Registros Polygon/MultiPolygon não vazios no ano",
      "deduplicacao":False,"reparacao":False,"reprojecao":False,"recorte_espacial":False,
      "versoes":{"python":platform.python_version(),"geopandas":gpd.__version__,"shapely":shapely.__version__,"geos":shapely.geos_version_string,"pandas":pd.__version__},
      "nota_hash":"Para SHP, o hash refere-se ao .shp; para ZIP refere-se ao pacote completo."}
(SAIDA/"metadados.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
# Arquiva somente os produtos desta execução.
zip_final=SAIDA.parent/"Auditoria_Geometrias_2021_2025.zip"
produtos=["resumo_anual.csv","tabela_para_word.html","texto_resultados.txt","motivos_invalidade_por_ano.csv",
          "registros_problematicos.csv","registros_fora_periodo.csv","metadados.json"]
with zipfile.ZipFile(zip_final,"w",zipfile.ZIP_DEFLATED) as z:
    for nome in produtos: z.write(SAIDA/nome,arcname=nome)
print("\nResultados:",zip_final)
if BAIXAR_ZIP:
    try:
        from google.colab import files
    except ImportError:
        print("Fora do Colab: acesse o ZIP na pasta acima.")
    else:
        files.download(str(zip_final))
